# Vanilla HNSW

In [1]:
import struct
import numpy as np
import torch
import torch.nn.functional as F
import heapq
from tqdm import tqdm

def read_fvecs(path: str) -> np.ndarray:
    """Read .fvecs file into (N, d) float32 array."""
    with open(path, 'rb') as f:
        vecs = []
        while True:
            hdr = f.read(4)
            if not hdr:
                break
            d = struct.unpack('i', hdr)[0]
            data = struct.unpack('f' * d, f.read(4 * d))
            vecs.append(data)
    return np.vstack(vecs).astype('float32')


def read_ivecs(path: str) -> np.ndarray:
    """Read .ivecs file into (N, k) int array."""
    with open(path, 'rb') as f:
        idxs = []
        while True:
            hdr = f.read(4)
            if not hdr:
                break
            k = struct.unpack('i', hdr)[0]
            data = struct.unpack('i' * k, f.read(4 * k))
            idxs.append(data)
    return np.vstack(idxs).astype('int64')

# ----------------------------
# Select GPU or CPU
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
display(device)

device(type='cuda')

In [2]:
class HNSWIndexTorch:
    def __init__(self, M=16, ef_construction=200, ef_search=200, metric='cosine', device=None):
        """
        M: max neighbors per node
        ef_construction: candidate pool size during construction
        ef_search: pool size during search
        metric: 'l2' or 'cosine'
        """
        self.M = M
        self.ef_construction = ef_construction
        self.ef_search = ef_search
        self.metric = metric
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # Will be initialized in fit()
        self.data = None  # torch.Tensor (N, d)
        self.levels = []  # list of dict: {'nodes': list[int], 'neighbors': {i: list[int]}}
        self.max_level = 0
        self.entry_point = None

    def _distance_batch(self, q: torch.Tensor, pts: torch.Tensor) -> torch.Tensor:
        # q: (d,), pts: (n, d)
        if self.metric == 'l2':
            return torch.norm(pts - q.unsqueeze(0), dim=1)
        # cosine distance
        qn = F.normalize(q.unsqueeze(0), dim=1)
        pn = F.normalize(pts, dim=1)
        return 1 - (qn @ pn.t()).squeeze(0)

    def fit(self, data_np: np.ndarray):
        """Build the HNSW index on GPU."""
        # load data to device
        pts = torch.from_numpy(data_np).to(self.device)
        if self.metric == 'cosine':
            pts = F.normalize(pts, dim=1)
        self.data = pts
        N, d = pts.shape
        # assign levels geom distributed
        levels = torch.from_numpy(
            np.random.geometric(1.0 - 1.0/self.M, size=N) - 1
        ).to(self.device)
        # for debug: pull to CPU so we can inspect it later
        self._levels = levels.cpu().numpy()
        self.max_level = int(levels.max().item())
        # init layers
        self.levels = []
        for _ in range(self.max_level + 1):
            self.levels.append({'nodes': [], 'neighbors': {}})
        # incremental insertion
        for i in tqdm(range(N), desc='HNSW build'):
            lvl = int(levels[i].item())
            for layer in range(lvl + 1):
                lyr = self.levels[layer]
                nodes = lyr['nodes']
                nbrs = lyr['neighbors']
                if not nodes:
                    nodes.append(i)
                    nbrs[i] = []
                    continue
                # compute distances to existing nodes in layer
                node_idx = torch.tensor(nodes, device=self.device)
                node_pts = pts[node_idx]
                qi = pts[i]
                dists = self._distance_batch(qi, node_pts)
                # select ef_construction smallest
                k = min(self.ef_construction, len(nodes))
                # pick the k smallest distances properly
                # (dists is a 1D tensor of length len(nodes))
                vals, inds = torch.topk(dists, k=k, largest=False)

                # now build the correct (node_id, dist) list
                cand = [
                    (int(node_idx[inds[j]].item()), float(vals[j].item()))
                    for j in range(k)
                ]

                # diversify select M
                selected = self._select_diverse(cand, self.M)
                nbrs[i] = selected
                # link back
                for j in selected:
                    lst = nbrs.get(j, [])
                    if len(lst) < self.M:
                        lst.append(i)
                    else:
                        # replace worst
                        dlist = [(jj, self._dist_scalar(jj, j)) for jj in lst]
                        worst_idx, _ = max(dlist, key=lambda x: x[1])
                        if self._dist_scalar(i, j) < self._dist_scalar(worst_idx, j):
                            lst.remove(worst_idx)
                            lst.append(i)
                    nbrs[j] = lst
                nodes.append(i)
        cent      = pts.mean(dim=0)
        top_nodes = self.levels[self.max_level]['nodes']           # only those in the top layer
        top_pts   = pts[torch.tensor(top_nodes, device=self.device)]
        d_top     = self._distance_batch(cent, top_pts)
        best_i    = int(torch.argmin(d_top).item())
        self.entry_point = top_nodes[best_i]


    def _dist_scalar(self, a: int, b: int) -> float:
        # scalar distance
        if self.metric == 'l2':
            return torch.norm(self.data[a] - self.data[b]).item()
        return float(1 - torch.dot(self.data[a], self.data[b]))

    def _select_diverse(self, candidates, M):
        """Greedy diversify from small candidate list."""
        selected = []
        for idx, dist in candidates:
            ok = True
            for s in selected:
                if self._dist_scalar(idx, s) < dist:
                    ok = False
                    break
            if ok:
                selected.append(idx)
                if len(selected) >= M:
                    break
        # pad if needed
        if len(selected) < M:
            for idx, _ in candidates:
                if idx not in selected:
                    selected.append(idx)
                    if len(selected) >= M:
                        break
        return selected

    def search(self, query_np: np.ndarray, k=10) -> list:
        q = torch.from_numpy(query_np).to(self.device)
        if self.metric == 'cosine':
            q = F.normalize(q, dim=0)

        ep = self.entry_point
        # 1) Greedy descent on upper layers
        for layer in range(self.max_level, 0, -1):
            nodes = self.levels[layer]['nodes']
            nbrmap = self.levels[layer]['neighbors']

            # Project entry-point if it isn’t in this layer
            if ep not in nodes:
                idxs = torch.tensor(nodes, device=self.device)
                dists = self._distance_batch(self.data[ep], self.data[idxs])
                # pick closest node in this layer
                best_local = int(idxs[ torch.argmin(dists).item() ].item())
                ep = best_local

            # now do the usual greedy step
            improved = True
            while improved:
                improved = False
                nbrs = nbrmap[ep]  # safe now
                if not nbrs:
                    break
                nbr_pts = self.data[nbrs]
                d_n = self._distance_batch(q, nbr_pts)
                best_i = torch.argmin(d_n).item()
                if d_n[best_i].item() < self._distance_batch(q, self.data[ep:ep+1]).item():
                    ep = nbrs[best_i]
                    improved = True

        # 2) Proper best‑first on level 0:
        visited = {ep}

        # candidate queue C: min‑heap of (dist, node)
        C = []
        # result queue W: max‑heap of (-dist, node) so that the largest dist is on top
        W = []

        d_ep = float(self._distance_batch(q, self.data[ep:ep+1]))
        heapq.heappush(C, (d_ep, ep))
        heapq.heappush(W, (-d_ep, ep))

        while C:
            dist_u, u = heapq.heappop(C)
            # worst distance in W
            worst_dist = -W[0][0]

            # pruning: if the next candidate is farther than our worst result, we’re done
            if len(W) >= self.ef_search and dist_u > worst_dist:
                break

            for v in self.levels[0]['neighbors'][u]:
                if v in visited:
                    continue
                visited.add(v)
                dv = float(self._distance_batch(q, self.data[v:v+1]))
                # if W isn’t full or v is better than the worst in W:
                if len(W) < self.ef_search or dv < worst_dist:
                    heapq.heappush(C, (dv, v))
                    heapq.heappush(W, (-dv, v))
                    # keep W at size ef_search
                    if len(W) > self.ef_search:
                        heapq.heappop(W)

            # update worst_dist for the next loop
            worst_dist = -W[0][0]

        # Extract nodes from W, sort by true distance, return top k
        result = [node for (_, node) in W]
        result.sort(key=lambda node: float(self._distance_batch(q, self.data[node:node+1])))
        return result[:k]

def recall_at_k(preds, gt, k=10):
    hits = 0
    n = len(preds)
    for i in range(n):
        hits += len(set(preds[i][:k]) & set(gt[i][:k]))
    print(hits, hits / n, hits / (n * k))
    return hits / (n * k)

In [3]:
if __name__ == '__main__':
    # paths
    base = read_fvecs('siftsmall/siftsmall_base.fvecs')
    query = read_fvecs('siftsmall/siftsmall_query.fvecs')
    gt = read_ivecs('siftsmall/siftsmall_groundtruth.ivecs')

    # at top of your script:
    # N = 1000
    # base = base[:N]
    # query = query[:100]
    # brute‑force L2 ground‑truth on the truncated base
    # (this gives you the “true” top‑10 neighbors within base[:N])
    dists = np.linalg.norm(
        query[:, None, :].astype('float32') 
      - base[None, :, :].astype('float32'), 
        axis=2
    )                   # shape (100, N)
    gt2 = np.argsort(dists, axis=1)[:, :10]  # (100, 10)
    idx = HNSWIndexTorch(M=32, ef_construction=200, ef_search=400, metric='l2', device=device)
    idx.fit(base)

    # 1) Level distribution
    unique, counts = np.unique(idx._levels, return_counts=True)
    print("level→count:", dict(zip(unique, counts)))

    # 2) Degree stats on level 0
    nbrs0 = idx.levels[0]['neighbors']
    degs = [len(neigh) for neigh in nbrs0.values()]
    print(f"layer0 degree: mean={np.mean(degs):.1f}, min={np.min(degs)}, max={np.max(degs)}")


    preds = []
    for q in tqdm(query, desc='Querying'):
        preds.append(idx.search(q, k=10))
    rec = recall_at_k(preds, gt2, k=10)
    print(f"Recall@10: {rec:.4f}")

    # for ef in [50, 100, 200, 400, 800]:
    #     idx = HNSWIndexTorch(
    #         M=16,
    #         ef_construction=200,
    #         ef_search=ef,
    #         metric='l2',
    #         device=device
    #     )
    #     idx.fit(base)
    #     preds = [idx.search(q, k=10) for q in query]
    #     rec = recall_at_k(preds, gt2, k=10)
    #     print(f"ef_search={ef:4d} → Recall@10 = {rec:.3f}")


HNSW build: 100%|██████████| 10000/10000 [07:03<00:00, 23.59it/s]


level→count: {0: 9690, 1: 302, 2: 7, 3: 1}
layer0 degree: mean=32.0, min=32, max=32


Querying: 100%|██████████| 100/100 [00:05<00:00, 17.94it/s]

1000 10.0 1.0
Recall@10: 1.0000
